We'll use window because it was better for LGBM

In [14]:
import pandas as pd
import numpy as np
from itertools import product
from lightgbm import LGBMClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,roc_auc_score, confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import random
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
set_seed(42)
scale=StandardScaler()
results=[]

In [15]:
#import dataframe
df_modelling=pd.read_parquet("../df_modelling.parquet")

In [16]:
#creating market vector
# 'Stock_Date', 'Closing Price($)', 'relative difference',
#        'comment_published_at_median_YMD', 'number_of_comments_at_date',
#        'comment_likes_agg', 'comment_embeddings_agg', 'comment_sentiment_agg',
#        'sentiment_tensor', 'videos_text_embeddings', 'TargetVariable:Up/Down'
numerical_columns=['Closing Price($) lag1',
'relative difference lag1',
         'number_of_comments_at_date',
    'comment_likes_agg',
    'comment_sentiment_agg',
     'sentiment_tensor']

print(df_modelling.columns)


#concating the embeddings separately
df_modelling['embeddings_vector']=df_modelling.apply(lambda row: np.concatenate((row['comment_embeddings_agg'], row['videos_text_embeddings_window'])), axis=1)
#concatenating the nummerical columns after that
#df_modelling['marketvector']=df_modelling.apply(lambda row: np.concatenate((row[numerical_columns].to_numpy(dtype=float),row['marketvector'])), axis=1)


Index(['Stock_Date', 'Closing Price($)', 'relative difference',
       'comment_published_at_YMD', 'number_of_comments_at_date',
       'comment_likes_agg', 'comment_embeddings_agg', 'comment_sentiment_agg',
       'sentiment_tensor', 'videos_text_embeddings',
       'videos_text_embeddings_window', 'TargetVariable:Up/Down',
       'Closing Price($) lag1', 'relative difference lag1'],
      dtype='object')


In [17]:
x=df_modelling['embeddings_vector'][0]
print(f"Shape of the market vector {x.shape}")

Shape of the market vector (512,)


In [18]:
#Defining X and y
y = df_modelling['TargetVariable:Up/Down'].values
numerical_values=np.stack(df_modelling[numerical_columns].values)
embeddings_values=np.stack(df_modelling['embeddings_vector'].values)
X =np.concatenate((numerical_values, embeddings_values), axis=1)

In [ ]:
def get_LGBM(X,y,n_estimators,learning_rate,num_leaves, max_depth, boosting_type, subsample, colsample_bytree):
    
    
    #defining a model
# Instantiate the LightGBM classifier
    model = LGBMClassifier(
    boosting_type=boosting_type,   # Gradient Boosting Decision Tree
    n_estimators=n_estimators,       # Number of trees
    learning_rate=learning_rate,      # Step size shrinkage
    max_depth=max_depth,           # No limit on tree depth
    num_leaves=num_leaves,          # Maximum number of leaves per tree
    random_state=42, 
    subsample=subsample,
    colsample_bytree=colsample_bytree
)
    #creating the splits for train, val and test

    n = X.shape[0]
    split1 = int(n * 0.7)       # 70% train
    split2 = int(n * 0.75)      # next 5% for threshold validation

    X_train = X[:split1]
    X_val   = X[split1:split2]
    X_test  = X[split2:]

    y_train = y[:split1]
    y_val   = y[split1:split2]
    y_test  = y[split2:]

    #training LGMB model
   
    model.fit(X_train, y_train)
    
    y_prob=model.predict_proba(X_test)[:,1]
    #making out the best threshold
    thresholds=np.arange(0.0,1.01,0.1)
    f1_scores=[]
    best_f1=0
    
    #target performance score macro avg of f1_score 
    y_val_prob = model.predict_proba(X_val)[:,1]
    for threshold in thresholds:
        y_prob_tr = (y_val_prob > threshold).astype(int)
        report=classification_report(y_val, y_prob_tr, output_dict=True)
        f1=report['macro avg']['f1-score']
        f1_scores.append(f1)
        #saving the best f1
        if f1 > best_f1:
            best_f1 = f1

    f1_score=max(f1_scores)
    index_tr=f1_scores.index(f1_score)
    threshold=thresholds[index_tr]
    y_test_prob = model.predict_proba(X_test)[:,1]
    y_prob_tr = (y_test_prob > threshold).astype(int)
    report=classification_report(y_test, y_prob_tr, output_dict=True)
    best_cm=confusion_matrix(y_test,y_prob_tr.astype(int))
    f1=report['macro avg']['f1-score']
    
    return f1,threshold, report, best_cm

    
    # print(classification_report(y_test, y_pred))
    # auc=roc_auc_score(y_test, y_prob)

    # fpr, tpr, _ = roc_curve(y_test, y_prob)

    # plt.figure(figsize=(6,6))
    # plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc:.2f})')
    # plt.plot([0, 1], [0, 1], 'k--')  # random line
    # plt.xlabel('False Positive Rate')
    # plt.ylabel('True Positive Rate')
    # plt.title('ROC Curve')
    # plt.legend(loc='lower right',bbox_to_anchor=(1,-0.2), facecolor='none')
    # plt.title("ROC for Logistic Regression")
    # #plt.savefig(f"ROC curve {str(text)}.png", bbox_inches='tight', transparent=True)
    # plt.show()



[LightGBM] [Info] Number of positive: 194, number of negative: 198
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005644 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 37966
[LightGBM] [Info] Number of data points in the train set: 392, number of used features: 518
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.494898 -> initscore=-0.020409
[LightGBM] [Info] Start training from score -0.020409
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

/Users/irtg/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/irtg/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [ ]:
#grid dictionary
param_grid = {
    "n_estimators": [100, 200, 300],       # number of boosting rounds
    "learning_rate": [0.01, 0.05, 0.1],   # step size shrinkage
    "num_leaves": [31, 50, 70],           # max number of leaves per tree
    "max_depth": [-1, 5, 10],             # -1 = no limit
    "boosting_type": ['gbdt', 'dart'],    # Gradient Boosting Decision Tree / Dropouts
    "subsample": [0.7, 0.8, 1.0],         # row sampling
    "colsample_bytree": [0.7, 0.8, 1.0]   # column sampling
}

In [ ]:
# Create all combinations of parameters
all_combinations = list(product(
    param_grid["n_estimators"],
    param_grid["learning_rate"],
    param_grid["num_leaves"],
    param_grid["max_depth"],
    param_grid["boosting_type"],
    param_grid["subsample"],
    param_grid["colsample_bytree"]
))

# Loop through each combination
for n_estimators, learning_rate, num_leaves, max_depth, boosting_type, subsample, colsample_bytree in all_combinations:
   f1_score,threshold, best_report, cm= get_LGBM(X,y,n_estimators,learning_rate,num_leaves, max_depth, boosting_type, subsample, colsample_bytree)
   results.append({
                        "n_estimators":n_estimators,
                        "learning_rate":learning_rate,
                        "num_leaves":num_leaves, 
                        "max_depth":max_depth, 
                        "boosting_type":boosting_type, 
                        "subsample":subsample,
                        "f1_score": f1_score,
                        "threshold":threshold,
                        "best_report":best_report, 
                        "confusion_matrix":cm
                    })

In [ ]:
df_results=pd.DataFrame(results)
df_results.to_csv("LGBM tuning window.csv")